<a href="https://colab.research.google.com/github/rahelDK/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#INSTALL & IMPORT DEPENDENCIES

In [2]:
!pip -q install pdfplumber sentence-transformers faiss-cpu transformers accelerate
import pdfplumber
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 21, in <module>
    from pip._internal.metadata import get_default_environment, get_environment
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/metadata/__init__.py", line 9, in <module>
    from .base import BaseDistribution, BaseEnvironment, FilesystemWheel, MemoryWheel, Wheel
  Fi

ModuleNotFoundError: No module named 'pdfplumber'

In [ ]:
from google.colab import files, drive
#uploaded = files.upload()
drive.mount('/content/drive')

In [ ]:

pdf_path = "/content/drive/MyDrive/10-K-2025-Apple.pdf"

#Extract Text From PDF

In [ ]:
def extract_text_from_pdf(pdf_file_path: str):
    pages_metadata = []
    full_text = []

    with pdfplumber.open(pdf_file_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = " ".join(text.split())
            pages_metadata.append({"page_number": i, "text": text})
            full_text.append(text)

    return "\n".join(full_text), pages_metadata


raw_text, pages_metadata = extract_text_from_pdf(pdf_path)
print("Pages extracted:", len(pages_metadata))
print("Sample (first 500 chars):", raw_text[:500])

#Chunk Text

In [ ]:
def chunk_text_with_metadata(pages_data, chunk_size=200, overlap=50):
    chunks = []
    step = chunk_size - overlap

    for page in pages_data:
        words = page["text"].split()
        if not words:
            continue

        for start in range(0, len(words), step):
            end = start + chunk_size
            chunk_words = words[start:end]
            if not chunk_words:
                continue

            chunks.append({
                "text": " ".join(chunk_words),
                "page_number": page["page_number"]
            })

    return chunks


text_chunks = chunk_text_with_metadata(pages_metadata, chunk_size=200, overlap=50)
print("Total chunks:", len(text_chunks))
print("Example chunk:", text_chunks[0]["page_number"], text_chunks[0]["text"][:250])

#CREATE EMBEDDINGS

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in text_chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True).astype("float32")


faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
print("Embedding dimension:", dimension)

#Build FAISS index

In [ ]:
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)

#Return top-k relevant chunks

In [ ]:
def retrieve_relevant_chunks(query: str, top_k: int = 3):
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)

    distances, indices = index.search(q_emb, top_k)
    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        results.append({
            "rank": rank + 1,
            "page_number": text_chunks[idx]["page_number"],
            "text": text_chunks[idx]["text"],
            "distance": float(distances[0][rank])
        })
    return results

#Load the generator model (FLAN-T5) + define RAG QA

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

def rag_qa_pipeline(
    query: str,
    top_k: int = 3,
    max_new_tokens: int = 200,
    temperature: float = 0.3,
    top_p: float = 0.9,
    top_k_gen: int = 50
):
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)
    context = "\n\n".join(
        [f"(Page {r['page_number']}) {r['text']}" for r in retrieved]
    )

    prompt = f"""You are answering ONLY using the context.
If the answer is not in the context, say: "I don't have enough information in the provided document."

Context:
{context}

Question: {query}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k_gen
    )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)


    cited_pages = sorted(set([r["page_number"] for r in retrieved]))
    return answer, retrieved, cited_pages

#TEST THE COMPLETE RAG SYSTEM

In [ ]:
question = "What are Apple’s total net sales for the fiscal year?"
answer, retrieved, cited_pages = rag_qa_pipeline(question, top_k=4)

print("ANSWER:\n", answer)
print("\nCITED PAGES:", cited_pages)

In [ ]:
question = "What subscription-based services does Apple offer?"
answer, retrieved, cited_pages = rag_qa_pipeline(question, top_k=4)

print("ANSWER:\n", answer)
print("\nCITED PAGES:", cited_pages)

In [ ]:
question = "What risks does Apple disclose regarding competition?"
answer, retrieved, cited_pages = rag_qa_pipeline(question, top_k=4)

print("ANSWER:\n", answer)
print("\nCITED PAGES:", cited_pages)